# EDA regional — Análisis A (producción por piso ecológico)

Exploración **descriptiva** de `dataset_regional.csv` (agregado región × piso × mes, cultivos Pareto-80 sumados).

**Objetivos (alineados al paper):**
1. Caracterizar volumen y estacionalidad productiva por unidad territorial.
2. Describir perfiles climáticos por piso ecológico.
3. Explorar co-movimientos clima–producción (Pearson, sin inferencia causal).
4. Documentar eventos extremos: sequía Puno 2022 y El Niño costero 2023–2024.

**Limitaciones explícitas:**
- Correlación ≠ causalidad; sin desestacionalización ni corrección Benjamini–Hochberg.
- Cultivos del mismo (región, piso, distrito) comparten clima idéntico.
- Producción en toneladas (volumen), no rendimiento t/ha.

**Unidades:** `radiacion_solar` MJ/m²/día; `precipitacion` mm/día; `humedad_suelo` índice 0–1.

**Downstream:** hallazgos regionales contextualizan `05_eda_por_cultivo.ipynb` y `06_clustering_cultivos.ipynb`.

### Qué hace esta celda

Configura el entorno: resuelve la ruta raíz del repositorio, crea `OUTPUTS/figures/`, carga `dataset_regional.csv` (1.008 filas = 14 unidades × 72 meses) y define constantes (`CLIMA_EDA`, orden de meses). Crea la columna `unidad` = región | piso | distrito y `fecha` para series temporales. Imprime dimensiones y conteo de NaN en producción.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent
elif not (ROOT / "OUTPUTS").exists() and (ROOT.parent / "OUTPUTS").exists():
    ROOT = ROOT.parent

RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)
RUTA_FIGURES = RUTA_OUTPUT / "figures"
RUTA_FIGURES.mkdir(parents=True, exist_ok=True)
DATASET_REGIONAL = RUTA_OUTPUT / "dataset_regional.csv"

if not DATASET_REGIONAL.exists():
    raise FileNotFoundError(
        f"No se encontró {DATASET_REGIONAL}. Ejecutar primero 03_build_dataset_integrado.ipynb"
    )

df = pd.read_csv(DATASET_REGIONAL)
df["unidad"] = (
    df["region"] + " | " + df["piso_ecologico"] + " (" + df["distrito"] + ")"
)
df["fecha"] = pd.to_datetime(
    df["anio"].astype(str) + "-" + df["numero_mes"].astype(str).str.zfill(2) + "-01"
)

CLIMA_EDA = [
    "temp_promedio", "precipitacion", "humedad_relativa",
    "radiacion_solar", "humedad_suelo",
]
MESES_ORDEN = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre",
]
df["mes"] = pd.Categorical(df["mes"], categories=MESES_ORDEN, ordered=True)

sns.set_theme(style="whitegrid", context="notebook")
print(f"Filas: {len(df):,} | Unidades: {df['unidad'].nunique()} | Regiones: {df['region'].nunique()}")
print(f"NaN produccion_piso_ton: {df['produccion_piso_ton'].isna().sum()}")
df.head(3)

## 1. Volumen productivo por unidad

Ranking de las 14 unidades (región × piso × distrito) según producción acumulada 2020–2025.

### Qué hace esta celda

Agrupa con `sum(min_count=1)` la producción de todos los cultivos Pareto-80 por unidad territorial (14 combinaciones región×piso×distrito) en el periodo 2020–2025. Calcula el porcentaje sobre el total nacional del subconjunto y genera un gráfico de barras horizontal guardado en `eda_regional_volumen_unidad.png`.

In [ ]:
vol_unidad = (
    df.groupby(["region", "piso_ecologico", "distrito", "unidad"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index(name="produccion_total_ton")
    .sort_values("produccion_total_ton", ascending=False)
)
vol_unidad["pct_total"] = 100 * vol_unidad["produccion_total_ton"] / vol_unidad["produccion_total_ton"].sum()

print("=== Top 5 unidades por volumen ===")
print(vol_unidad.head(5)[["unidad", "produccion_total_ton", "pct_total"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=vol_unidad, y="unidad", x="produccion_total_ton",
    hue="region", dodge=False, legend=False, ax=ax,
)
ax.set_title("Producción acumulada 2020–2025 por unidad (Pareto-80)")
ax.set_xlabel("Toneladas")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_volumen_unidad.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — volumen por unidad (`eda_regional_volumen_unidad.png`)

Este gráfico responde: **¿dónde se concentra el volumen productivo del subconjunto Pareto-80?**

- Cada barra es una **unidad territorial** (región + piso ecológico + distrito NASA), no una región administrativa completa.
- Las unidades con barras más largas (típicamente **costa de La Libertad — Virú**, **Ica — Chincha**, **selva de San Martín o Junín**) dominan el volumen porque albergan cultivos de alto tonelaje (caña, arroz, palta, espárrago, etc.).
- Los pisos de **altiplano (Puno)** suelen aparecer con barras menores en toneladas absolutas, pero no son menos relevantes agronómicamente: reflejan escala de cultivos andinos y filtro Pareto, no "menor importancia".
- El `% del total` en la tabla impresa cuantifica la concentración: pocas unidades explican gran parte del volumen del análisis.

**Lectura honesta:** estamos sumando **toneladas**, no rendimiento (t/ha). Una barra grande puede deberse a extensión sembrada o a cultivos industrializados, no necesariamente a mayor productividad por hectárea.

## 2. Series temporales de producción

Agregación anual y mensual; líneas por unidad territorial.

### Qué hace esta celda

Construye dos paneles: (arriba) producción **anual** por cada una de las 14 unidades; (abajo) producción **mensual** sumando todos los pisos dentro de cada región. Guarda `eda_regional_produccion_anual.png`.

In [ ]:
prod_anual = (
    df.groupby(["region", "piso_ecologico", "distrito", "unidad", "anio"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

for unidad, g in prod_anual.groupby("unidad"):
    axes[0].plot(g["anio"], g["produccion_piso_ton"], marker="o", label=unidad, linewidth=1.2)
axes[0].set_title("Producción anual agregada por unidad")
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Toneladas")
axes[0].legend(bbox_to_anchor=(1.02, 1), fontsize=7, ncol=1)

for region, g in df.groupby("region"):
    m = g.groupby("fecha", observed=True)["produccion_piso_ton"].sum(min_count=1)
    axes[1].plot(m.index, m.values, label=region, linewidth=1.2)
axes[1].set_title("Producción mensual sumada por región (todas las unidades)")
axes[1].set_ylabel("Toneladas")
axes[1].legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_produccion_anual.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — series temporales (`eda_regional_produccion_anual.png`)

**Panel superior (anual por unidad):**
- Cada línea = una de las 14 unidades. Permite ver tendencias interanuales y comparar pisos dentro de la misma región (ej. Junín tiene selva alta, selva baja y sierra con trayectorias distintas).
- Caídas marcadas en **Puno 2022** son compatibles con la sequía altiplánica documentada en el paper; subidas en años favorables reflejan recuperación o años hidrológicamente mejores.
- Picos aislados pueden deberse a meses MIDAGRI imputados como NaN en el agregado (`min_count=1` evita falsos ceros).

**Panel inferior (mensual por región):**
- Suma todos los pisos de cada región → visión macro de estacionalidad y shocks.
- La **estacionalidad** se ve como oscilaciones repetidas cada año; desviaciones sostenidas en 2022–2023 en sur andino o anomalías en costa 2023–2024 enlazan con los casos de estudio posteriores.

**No interpretar como:** efecto causal del clima mes a mes; es co-movimiento temporal producción agregada.

## 3. Estacionalidad (mes × año)

Heatmaps por región: intensidad de cosecha mensual a lo largo del periodo.

### Qué hace esta celda

Para cada una de las 6 regiones, arma una tabla mes×año con la producción agregada y la visualiza como heatmap (`YlOrRd`). Permite ver en qué meses y años hubo picos o caídas de cosecha. Guarda `eda_regional_heatmap_estacionalidad.png`.

In [ ]:
regiones = sorted(df["region"].unique())
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for ax, region in zip(axes, regiones):
    sub = df[df["region"] == region]
    pivot = sub.pivot_table(
        index="mes", columns="anio", values="produccion_piso_ton",
        aggfunc=lambda x: x.sum(min_count=1), observed=True,
    )
    pivot = pivot.reindex(MESES_ORDEN)
    sns.heatmap(pivot, ax=ax, cmap="YlOrRd", cbar_kws={"label": "ton"}, linewidths=0.2)
    ax.set_title(region)
    ax.set_xlabel("Año")
    ax.set_ylabel("Mes")

fig.suptitle("Estacionalidad productiva — mes × año (agregado regional)", y=1.02)
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_heatmap_estacionalidad.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — heatmaps estacionalidad (`eda_regional_heatmap_estacionalidad.png`)

Cada subpanel es una **región**; filas = meses, columnas = años; color = toneladas producidas ese mes.

- **Celdas más intensas (rojo):** meses de mayor cosecha agregada en esa región (calendario productivo principal).
- **Costa norte (Piura, La Libertad, Ica):** a menudo patrones más continuos o concentrados en campañas de riego/verano según cultivo dominante.
- **Sierra y altiplano (Puno, pisos de Junín/La Libertad):** estacionalidad más marcada; celdas pálidas en invierno seco andino.
- **Selva (San Martín, Junín baja):** puede mostrar producción más distribuida o picos ligados a ciclos de cultivos perennes/transitorios del Pareto.
- **Celdas vacías o muy claras:** mes sin producción reportada (NaN en origen) o mes fuera de ventana de cosecha.
- Comparar **2022 vs años vecinos** en Puno/Junín sierra ayuda a visualizar el evento de sequía sin calcular elasticidades.

Este gráfico sustenta en el paper la "caracterización de patrones de estacionalidad" a nivel regional agregado.

## 4. Patrón estacional promedio

Promedio mensual de producción (todos los años) por unidad.

### Qué hace esta celda

Promedia la producción de cada mes del calendario a través de todos los años (enero–diciembre) por unidad. Obtiene el **calendario agrícola típico** de cada piso. Guarda `eda_regional_patron_estacional.png`.

In [ ]:
patron = (
    df.groupby(["unidad", "mes"], observed=True)["produccion_piso_ton"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
for unidad, g in patron.groupby("unidad"):
    g = g.sort_values("mes")
    ax.plot(g["mes"].astype(str), g["produccion_piso_ton"], marker="o", label=unidad, linewidth=1)
ax.set_title("Patrón estacional promedio por unidad")
ax.set_xlabel("Mes")
ax.set_ylabel("Toneladas (promedio mensual)")
ax.tick_params(axis="x", rotation=45)
ax.legend(bbox_to_anchor=(1.02, 1), fontsize=7)
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_patron_estacional.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — patrón estacional promedio (`eda_regional_patron_estacional.png`)

Promedia todos los años → curva **típica** mes a mes por unidad (suaviza shocks como 2022).

- **Picos de curva:** meses de cosecha habitual (ej. verano costero, post-lluvias en sierra).
- **Valles:** meses sin cosecha o con producción marginal.
- Unidades de la misma región pero distinto piso (ej. La Libertad: costa vs sierra vs yunga) deben mostrar curvas **desfasadas o de distinta amplitud**, validando el enfoque multi-distrito del proyecto frente a un solo punto climático por departamento.
- Si dos unidades tienen curvas casi idénticas en producción pero climas distintos, recuerda que aquí la producción está **sumada por cultivo Pareto** y el clima no entra aún en este gráfico.

Útil para la defensa: explica "cuándo se cosecha" antes de hablar de correlaciones clima–producción.

## 5. Perfil climático por piso

Medias 2020–2025 de variables climáticas core por unidad (descriptivo).

### Qué hace esta celda

Calcula medias 2020–2025 de las 5 variables climáticas core por unidad (panel de barras) y un boxplot de temperatura mensual por región y piso. Guarda `eda_regional_perfil_climatico.png` y `eda_regional_boxplot_temp.png`.

In [ ]:
clima_mean = (
    df.groupby(["unidad", "region"], observed=True)[CLIMA_EDA]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, len(CLIMA_EDA), figsize=(16, 5))
for ax, var in zip(axes, CLIMA_EDA):
    sns.barplot(
        data=clima_mean.sort_values(var, ascending=False),
        x=var, y="unidad", hue="region", dodge=False, ax=ax, legend=False,
    )
    ax.set_title(var)
    ax.set_ylabel("")
fig.suptitle("Perfil climático promedio por unidad (2020–2025)", y=1.02)
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_perfil_climatico.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x="region", y="temp_promedio", hue="piso_ecologico", ax=ax)
ax.set_title("Distribución mensual de temperatura por región y piso")
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_boxplot_temp.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — perfil climático (`eda_regional_perfil_climatico.png`, `eda_regional_boxplot_temp.png`)

**Panel de barras (5 variables):**
- Resume el **clima promedio 2020–2025** del punto NASA de cada distrito.
- `temp_promedio`: gradiente costa cálida → altiplano frío (Puno/Ayaviri, Ilave más bajos).
- `precipitacion`: contraste costa árida (Ica, Virú) vs sierra/selva húmeda (Canchaque, Moyobamba, Perené).
- `humedad_suelo` / `humedad_relativa`: indicadores de estrés hídrico; valores bajos sostenidos en costa y puna.
- `radiacion_solar`: alta en costa despejada; modulada en selva nublada.

**Boxplot de temperatura:**
- Muestra **variabilidad mensual** (no solo la media): amplitud térmica intra-anual por piso.
- Pisos de sierra y altiplano suelen tener cajas más bajas y a veces mayor dispersión estacional.

**Limitación:** un solo punto por distrito; no captura microclima dentro del valle o la cuenca.

## 6. Correlaciones exploratorias (Pearson)

Asociación mensual clima–producción **por unidad**. Sin corrección por comparaciones múltiples.

### Qué hace esta celda

Calcula correlación de Pearson entre producción mensual y cada variable climática **dentro** de cada unidad (mínimo 12 meses válidos). Exporta `eda_correlaciones_regional.csv` e imprime el top-5 de |r| y un pooling agregado (todas las unidades juntas). **No implica causalidad** ni corrección por comparaciones múltiples.

In [ ]:
rows = []
for unidad, g in df.groupby("unidad"):
    sub = g.dropna(subset=["produccion_piso_ton"] + CLIMA_EDA)
    if len(sub) < 12:
        continue
    region = g["region"].iloc[0]
    piso = g["piso_ecologico"].iloc[0]
    distrito = g["distrito"].iloc[0]
    for var in CLIMA_EDA:
        r, p = stats.pearsonr(sub["produccion_piso_ton"], sub[var])
        rows.append({
            "unidad": unidad, "region": region, "piso_ecologico": piso,
            "distrito": distrito, "variable_clima": var,
            "n": len(sub), "r": r, "p_valor": p,
        })

corr_regional = pd.DataFrame(rows)
corr_regional.to_csv(RUTA_OUTPUT / "eda_correlaciones_regional.csv", index=False, encoding="utf-8-sig")

top5 = corr_regional.reindex(corr_regional["r"].abs().sort_values(ascending=False).index).head(5)
print("=== Top 5 |r| por unidad (exploratorio, sin BH) ===")
print(top5[["unidad", "variable_clima", "n", "r", "p_valor"]].to_string(index=False))

sub_all = df.dropna(subset=["produccion_piso_ton"] + CLIMA_EDA)
print("\n=== Correlación agregada (pooling de todas las unidades) ===")
for var in CLIMA_EDA:
    r, p = stats.pearsonr(sub_all["produccion_piso_ton"], sub_all[var])
    print(f"{var:20s} r={r:+.3f}  p={p:.2e}  n={len(sub_all)}")

### Interpretación — correlaciones Pearson (tablas y CSV)

Cada fila del CSV = una unidad × una variable climática; `r` mide asociación lineal mensual.

- **|r| alto** (ej. > 0,6): meses con más producción tienden a coincidir con meses con valores altos/bajos de esa variable en esa unidad — **no** demuestra que el clima "cause" la cosecha (confusión por estacionalidad compartida).
- **Signo de r:** positivo = ambas suben juntas; negativo = una sube cuando la otra baja.
- El **pooling agregado** (todas las unidades mezcladas) mezcla climas muy distintos; un r global (ej. temperatura) puede ser débil o engañoso. Prioriza siempre el análisis **por unidad** o el notebook 05 por cultivo.
- Sin corrección Benjamini–Hochberg: con 14×5 = 70 contrastes, algunos p_valor < 0,05 pueden ser falsos positivos.

En el paper se menciona temperatura como variable influyente a nivel agregado; verifica que la cifra citada corresponda al mismo nivel de agregación y periodo que esta tabla.

## 7. Caso sequía — Puno 2021→2022

Co-movimiento descriptivo entre producción agregada y humedad de suelo en altiplano (no atribución causal).

### Qué hace esta celda

Filtra Puno, agrega producción anual y humedad de suelo media por unidad, calcula el % de cambio 2021→2022 y grafica en ejes duales producción vs humedad radicular. Guarda `eda_regional_puno_sequia.png`.

In [ ]:
puno = df[df["region"] == "Puno"].copy()

prod_puno = (
    puno.groupby(["unidad", "anio"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index()
)
clima_puno = (
    puno.groupby(["unidad", "anio"], observed=True)["humedad_suelo"]
    .mean()
    .reset_index()
)

cambios = []
for unidad in prod_puno["unidad"].unique():
    p21 = prod_puno.query("unidad == @unidad and anio == 2021")["produccion_piso_ton"]
    p22 = prod_puno.query("unidad == @unidad and anio == 2022")["produccion_piso_ton"]
    if len(p21) and len(p22) and p21.iloc[0] > 0:
        pct = 100 * (p22.iloc[0] - p21.iloc[0]) / p21.iloc[0]
        cambios.append({"unidad": unidad, "pct_cambio_2021_2022": round(pct, 1)})
if cambios:
    print("=== Cambio producción Puno 2021→2022 ===")
    print(pd.DataFrame(cambios).to_string(index=False))

fig, ax1 = plt.subplots(figsize=(9, 4))
for unidad, g in prod_puno.groupby("unidad"):
    ax1.plot(g["anio"], g["produccion_piso_ton"], marker="o", label=f"Prod. {unidad}")
ax1.set_ylabel("Producción (ton)")
ax1.set_title("Puno — producción anual vs humedad de suelo (sequía altiplánica 2022)")
ax1.legend(fontsize=7, loc="upper left")

ax2 = ax1.twinx()
for unidad, g in clima_puno.groupby("unidad"):
    ax2.plot(g["anio"], g["humedad_suelo"], marker="s", linestyle="--", alpha=0.7, label=f"HS {unidad}")
ax2.set_ylabel("Humedad suelo (índice)")
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_puno_sequia.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — sequía Puno (`eda_regional_puno_sequia.png`)

- **Líneas sólidas (eje izquierdo):** producción anual agregada en Ilave y Ayaviri (altiplano).
- **Líneas punteadas (eje derecho):** humedad de zona radicular (`humedad_suelo`, índice 0–1 de NASA POWER).
- Una **caída fuerte 2021→2022** en producción junto a humedad de suelo baja o estancada es **evidencia descriptiva** coherente con sequía altiplánica; no sustituye un estudio causal con rendimiento (t/ha) ni control de área sembrada.
- La tabla impresa de `% cambio` cuantifica la magnitud para cada unidad (cultivos andinos agregados en Pareto).
- Si una unidad recupera en 2023–2024, puede reflejar año hidrológico menos severo, no necesariamente política o tecnología.

Para el detalle por cultivo (quinua, papa, etc.) ver `05_eda_por_cultivo.ipynb`.

## 8. Caso El Niño costero 2023–2024

Anomalías de precipitación y temperatura en Piura, La Libertad e Ica respecto al promedio 2020–2022.

### Qué hace esta celda

Compara precipitación y temperatura en Piura, La Libertad e Ica entre la referencia 2020–2022 y los años 2023–2024 (ventana El Niño costero). Grafica series mensuales de precipitación y producción con sombreado 2023–2024. Guarda `eda_regional_nino_costa.png`.

In [ ]:
costa = df[df["region"].isin(["Piura", "La Libertad", "Ica"])].copy()
ref = costa[costa["anio"].between(2020, 2022)].groupby("region")[["precipitacion", "temp_promedio"]].mean()
evt = costa[costa["anio"].isin([2023, 2024])].groupby(["region", "anio"])[["precipitacion", "temp_promedio"]].mean()

print("=== Referencia 2020–2022 vs evento 2023–2024 (costa norte-centro) ===")
for region in ["Piura", "La Libertad", "Ica"]:
    if region not in ref.index:
        continue
    ref_p = ref.loc[region, "precipitacion"]
    ref_t = ref.loc[region, "temp_promedio"]
    print(f"\n{region} — ref precip={ref_p:.3f}  temp={ref_t:.2f}")
    if region in evt.index.get_level_values(0):
        print(evt.loc[region].round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for region, g in costa.groupby("region"):
    m = g.groupby("fecha", observed=True)["precipitacion"].mean()
    axes[0].plot(m.index, m.values, label=region, linewidth=1.2)
axes[0].axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31"), alpha=0.15, color="coral", label="2023–2024")
axes[0].set_title("Precipitación mensual — costa")
axes[0].set_ylabel("mm/día")
axes[0].legend(fontsize=8)

for region, g in costa.groupby("region"):
    m = g.groupby("fecha", observed=True)["produccion_piso_ton"].sum(min_count=1)
    axes[1].plot(m.index, m.values, label=region, linewidth=1.2)
axes[1].axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31"), alpha=0.15, color="coral")
axes[1].set_title("Producción mensual agregada — costa")
axes[1].set_ylabel("Toneladas")
axes[1].legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_nino_costa.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretación — El Niño costero (`eda_regional_nino_costa.png`)

- **Panel izquierdo:** precipitación mensual media en Piura, La Libertad e Ica; zona sombreada = 2023–2024.
- Durante episodios El Niño costero suele observarse **precipitación anómala** (aumento en norte, posible estrés en cultivos sensibles a exceso hídrico o calor).
- **Panel derecho:** producción mensual agregada en las mismas regiones; busca desfases (el impacto productivo puede aparecer meses después del pico pluvial).
- La tabla impresa compara medias 2020–2022 vs 2023–2024 por región y año.

**Cautela:** El Niño afecta de forma heterogénea por cultivo (arroz vs uva vs caña); este gráfico es regional agregado. No atribuir al clima sin desagregar.

## 9. Auditoría de calidad y cierre

- NaN en producción: meses sin cosecha reportada (política del pipeline, no imputados).
- **Siguiente paso:** `05_eda_por_cultivo.ipynb` desagrega por cultivo; `06_clustering_cultivos.ipynb` construye tipologías sobre perfiles Pareto-80.

### Qué hace esta celda

Audita NaN en columnas climáticas y producción; imprime un resumen cuantitativo del Análisis A (filas, unidades, figuras generadas). Cierra el notebook con enlace a 05 y 06.

In [ ]:
clima_cols = [c for c in df.columns if c.startswith("temp_") or c in (
    "precipitacion", "humedad_relativa", "radiacion_solar", "humedad_suelo",
)]
nan_audit = df[clima_cols + ["produccion_piso_ton"]].isna().sum()
print("=== Auditoría NaN ===")
print(nan_audit[nan_audit > 0] if (nan_audit > 0).any() else "Sin NaN en columnas revisadas")

resumen = {
    "filas": len(df),
    "unidades": df["unidad"].nunique(),
    "regiones": df["region"].nunique(),
    "nan_produccion": int(df["produccion_piso_ton"].isna().sum()),
    "figuras_generadas": len(list(RUTA_FIGURES.glob("eda_regional_*.png"))),
}
print("\n=== Resumen Análisis A ===")
for k, v in resumen.items():
    print(f"{k}: {v}")

### Cierre del Análisis A

Con estos nueve bloques se cubren los objetivos regionales del paper: volumen, estacionalidad, clima, asociaciones exploratorias y dos eventos extremos. El siguiente paso lógico es **desagregar por cultivo** (`05`) y luego **tipologías** (`06`).